In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# Caminho base do Google Drive
drive_base_path = '/content/drive/My Drive'

# Primeiro caminho: pasta de imagens
images_path = os.path.join(drive_base_path, 'sistematizacao-modelos-multimodais', 'atelie-generativo-brasilia-arquitetura', 'dados', 'imagens')

print(f"\nTentando acessar: {images_path}")
if os.path.exists(images_path):
    print(f"Conteúdo de '{images_path}':")
    for item in os.listdir(images_path):
        print(item)
else:
    print(f"O caminho '{images_path}' não foi encontrado. Verifique se o caminho está correto e se os arquivos existem.")


Tentando acessar: /content/drive/My Drive/sistematizacao-modelos-multimodais/atelie-generativo-brasilia-arquitetura/dados/imagens
Conteúdo de '/content/drive/My Drive/sistematizacao-modelos-multimodais/atelie-generativo-brasilia-arquitetura/dados/imagens':
id_imagem_001.jpg
id_imagem_002.jpg
id_imagem_003.jfif
id_imagem_004.jpg
id_imagem_005.jpg
id_imagem_006.jpg
id_imagem_007.jpg
id_imagem_008.jpg
id_imagem_009.jpg
id_imagem_010.jpg
id_imagem_011.jpg
id_imagem_012.jfif
id_imagem_013.jpg
id_imagem_014.jpg
id_imagem_015.jpg
id_imagem_016.jpg
id_imagem_017.jpg
id_imagem_018.jpg
id_imagem_019.jpg
id_imagem_020.jpg
id_imagem_021.jpg
id_imagem_022.jpg
id_imagem_023.jpg
id_imagem_024.jpg
id_imagem_025.jpg
id_imagem_026.jpg
id_imagem_027.jpg
id_imagem_028.jfif
id_imagem_029.jpg
id_imagem_030.jpg
id_imagem_031.jpg
id_imagem_032.jpg
id_imagem_033.jpg
id_imagem_034.jpg
id_imagem_035.jpg
id_imagem_036.jfif
id_imagem_037.jfif
id_imagem_038.jpg
id_imagem_039.jpg
id_imagem_040.jpg
id_imagem_041.jfi

In [ ]:
# Segundo caminho: arquivo legendas.txt
labels_file_path = os.path.join(drive_base_path, 'sistematizacao-modelos-multimodais', 'atelie-generativo-brasilia-arquitetura', 'dados', 'legendas.txt')

print(f"\nTentando acessar: {labels_file_path}")
if os.path.exists(labels_file_path):
    print(f"O arquivo '{labels_file_path}' existe e pode ser acessado.")
    # Exemplo de como ler o arquivo:
    # with open(labels_file_path, 'r') as f:
    #     content = f.read()
    #     print("Conteúdo (primeiras 100 caracteres):\n", content[:100])
else:
    print(f"O arquivo '{labels_file_path}' não foi encontrado. Verifique se o caminho está correto e se o arquivo existe.")


Tentando acessar: /content/drive/My Drive/sistematizacao-modelos-multimodais/atelie-generativo-brasilia-arquitetura/dados/legendas.txt
O arquivo '/content/drive/My Drive/sistematizacao-modelos-multimodais/atelie-generativo-brasilia-arquitetura/dados/legendas.txt' existe e pode ser acessado.


In [ ]:
import os
import shutil
import csv

# Caminhos originais do seu Google Drive
drive_images_path = '/content/drive/My Drive/sistematizacao-modelos-multimodais/atelie-generativo-brasilia-arquitetura/dados/imagens'
drive_labels_path = '/content/drive/My Drive/sistematizacao-modelos-multimodais/atelie-generativo-brasilia-arquitetura/dados/legendas.txt'

# Pasta local de destino que o script do Hugging Face vai ler
local_dataset_path = '/content/dataset'
os.makedirs(local_dataset_path, exist_ok=True)

# Mapear os arquivos reais na pasta do Drive para contornar a questão do .jpg vs .jfif
arquivos_no_drive = os.listdir(drive_images_path)
mapa_arquivos_reais = {os.path.splitext(f)[0]: f for f in arquivos_no_drive}

metadata_rows = []

print("Iniciando o loop de cópia e estruturação do dataset...")

# Ler o arquivo de legendas original
with open(drive_labels_path, 'r', encoding='utf-8') as f:
    for linha in f:
        linha = linha.strip()
        if not linha or '|' not in linha:
            continue

        # Separar o nome do arquivo da legenda pelo pipe '|'
        nome_arquivo_txt, legenda = linha.split('|', 1)
        nome_base = os.path.splitext(nome_arquivo_txt)[0]

        # Verificar se o arquivo existe no Drive (independente de ser .jpg ou .jfif)
        if nome_base in mapa_arquivos_reais:
            nome_real = mapa_arquivos_reais[nome_base]
            caminho_origem = os.path.join(drive_images_path, nome_real)
            caminho_destino = os.path.join(local_dataset_path, nome_real)

            # Copia a imagem do Drive para o ambiente local do Colab (mais rápido para o treino)
            shutil.copy(caminho_origem, caminho_destino)

            # Adiciona ao mapeamento do metadata.csv (esperado pelo diffusers: file_name e text)
            metadata_rows.append({
                'file_name': nome_real,
                'text': legenda
            })
        else:
            print(f"Aviso: A imagem '{nome_arquivo_txt}' listada no txt não foi encontrada na pasta do Drive.")

# Gravar o arquivo metadata.csv exigido pelo Hugging Face Datasets
metadata_file_path = os.path.join(local_dataset_path, 'metadata.csv')
with open(metadata_file_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['file_name', 'text'])
    writer.writeheader()
    writer.writerows(metadata_rows)

print(f"\nSucesso! {len(metadata_rows)} imagens copiadas e 'metadata.csv' gerado em: {local_dataset_path}")

Iniciando o loop de cópia e estruturação do dataset...

Sucesso! 41 imagens copiadas e 'metadata.csv' gerado em: /content/dataset


In [ ]:
# 1. Instalar as dependências necessárias
!pip -q install diffusers transformers accelerate peft datasets

In [ ]:
# 2. Clonar o repositório do Diffusers e navegar até o diretório do script
!git clone https://github.com/huggingface/diffusers
%cd diffusers/examples/text_to_image

Cloning into 'diffusers'...
remote: Enumerating objects: 126096, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (140/140), done.
remote: Total 126096 (delta 82), reused 30 (delta 22), pack-reused 125932 (from 2)
Receiving objects: 100% (126096/126096), 99.48 MiB | 19.25 MiB/s, done.
Resolving deltas: 100% (94067/94067), done.
/content/diffusers/examples/text_to_image


In [ ]:
from huggingface_hub import login
login()

In [ ]:
# Desinstala a versão padrão e instala a versão de desenvolvimento direto do GitHub
!pip uninstall -y diffusers
!pip install git+https://github.com/huggingface/diffusers

Found existing installation: diffusers 0.38.0
Uninstalling diffusers-0.38.0:
  Successfully uninstalled diffusers-0.38.0
  Cloning https://github.com/huggingface/diffusers to /tmp/pip-req-build-qztbfds3
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers /tmp/pip-req-build-qztbfds3
  Resolved https://github.com/huggingface/diffusers to commit cbdb63798bc627e35ca6656265b7185e7618ad92
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for diffusers: filename=diffusers-0.39.0.dev0-py3-none-any.whl size=5631249 sha256=9e86d99dda590f827c10975bf907db1f512ab666e55338aff63db850e90c36df
  Stored in directory: /tmp/pip-ephem-wheel-cache-f5qcbx92/wheels/90/d4/44/a58bc00fb405fefb633b0d9d2307f6e3aec6cc1775d82555d3
Successfully built diffusers


In [ ]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 45.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="/content/dataset" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1500 \
  --learning_rate=1e-4 \
  --lr_scheduler="cosine" \
  --rank=8 \
  --mixed_precision="fp16" \
  --checkpointing_steps=500 \
  --validation_prompt="estilo_arquitetura_brasilia, o Congresso Nacional sob um céu azul limpo" \
  --output_dir="/content/lora_saida_config1" \
  --push_to_hub \
  --hub_model_id="wesleymqsss/lora-estilo-arquitetura-brasilia-r8"

Streaming output truncated to the last 5000 lines.
07/04/2026 21:41:03 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/stable-diffusion-v1-5/stable-diffusion-v1-5/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/model_index.json "HTTP/1.1 200 OK"
{'requires_safety_checker', 'image_encoder'} was not found in config. Values will be initialized to default values.

Loading pipeline components...:   0% 0/7 [00:00<?, ?it/s]Loaded feature_extractor as CLIPImageProcessor from `feature_extractor` subfolder of stable-diffusion-v1-5/stable-diffusion-v1-5.


Loading weights: 100% 196/196 [00:00<00:00, 2042.67it/s]
Loaded text_encoder as CLIPTextModel from `text_encoder` subfolder of stable-diffusion-v1-5/stable-diffusion-v1-5.

Loading pipeline components...:  43% 3/7 [00:00<00:00, 14.60it/s]{'prediction_type', 'timestep_spacing'} was not found in config. Values will be initialized to default values.
Loaded scheduler as PNDMScheduler from `scheduler` subfolder of stable

In [ ]:
!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="/content/dataset" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1500 \
  --learning_rate=5e-5 \
  --lr_scheduler="cosine" \
  --rank=4 \
  --mixed_precision="fp16" \
  --checkpointing_steps=500 \
  --validation_prompt="estilo_arquitetura_brasilia, o Congresso Nacional sob um céu azul limpo" \
  --output_dir="/content/lora_saida_config2" \
  --push_to_hub \
  --hub_model_id="wesleymqsss/lora-estilo-arquitetura-brasilia-r4"

Streaming output truncated to the last 5000 lines.


Loading weights:   0% 0/196 [00:00<?, ?it/s]

Loading weights: 100% 196/196 [00:00<00:00, 1282.87it/s]
Loaded text_encoder as CLIPTextModel from `text_encoder` subfolder of stable-diffusion-v1-5/stable-diffusion-v1-5.

Loading pipeline components...:  71% 5/7 [00:01<00:00,  5.02it/s]Loaded tokenizer as CLIPTokenizer from `tokenizer` subfolder of stable-diffusion-v1-5/stable-diffusion-v1-5.
Loaded feature_extractor as CLIPImageProcessor from `feature_extractor` subfolder of stable-diffusion-v1-5/stable-diffusion-v1-5.
Loading pipeline components...: 100% 7/7 [00:01<00:00,  5.57it/s]
07/04/2026 22:54:44 - INFO - __main__ - [RANK 0] Running validation... 
 Generating 4 images with prompt: estilo_arquitetura_brasilia, o Congresso Nacional sob um céu azul limpo.
Steps:  14% 209/1500 [13:48<33:28,  1.56s/it, lr=2.09e-5, step_loss=0.256]07/04/2026 22:55:28 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/models/stable-diffusion